# Pre-processing MultiplEYE Data

This notebook provides a step-by-step guide through how to process the eye-tracking data and the psychometric tests data collected within the MultiplEYE project. This goal of this notebook is twofold:

1. To provide a step-by-step guide on how to preprocess MultiplEYE data using the `pymovements` library and our custom preprocessing functions.
2. To serve as a tutorial for researchers who want to preprocess their own MultiplEYE data, or data from other eye-tracking datasets, using the `pymovements` library.

## Preparation steps
1. Download the data folder from the online repository. Note that this is only possible if you have access to at least one data collection protected folder. You will have access if you are an active member of one data collection group. Download the entire content of the folder.
When you download it from SwitchDrive, it will automatically create a .tar file.
2. Add the folder to the `data/` folder in this repo. The name of the folder is the data collection name, e.g., `MultiplEYE_ZH_CH_Zurich_1_2025`.
3. Extract the .tar file in the `data/` folder.
4. Make sure that the folder structure is correct. It should look like the one online and like this (there might be more data but this is not relevant at this point):
```
	MultiplEYE_ZH_CH_Zurich_1_2025/
		documentation/
		eye-tracking-sessions/
			001_.../
			002_.../
			...
			pilot_sessions/
				001_.../
				002_.../
				...
		psychometric-tests-sessions/
		stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/
		...
```

## The config file



The pipeline uses a config file which can be used to specify parameters and settings for the preprocessing. It is typically named `multipleye_settings_preprocessing.yaml`. You can load it explicitly or rely on the default loading mechanism (CWD, environment variable, or legacy root).

Once you have your config file ready, you can load it as shown below.

In [1]:
# from preprocessing.data_collection.multipleye_data_collection import prepare_language_folder
from preprocessing.data_collection.multipleye_data_collection import (
    MultipleyeDataCollection,
)

import preprocessing

# the settings will be loaded into general config module, so we can access all settings at the same place
from preprocessing import settings

from preprocessing.scripts.prepare_language_folder import prepare_language_folder

import polars as pl

2026-09-10 16:28:39,147 - preprocessing - INFO - Pipeline version: 2026.8.28
2026-09-10 16:28:39,149 - preprocessing - INFO - Last updated (git): v2026.08.28-19-ge8d068d (2026-09-08 16:52:28 +0200)
2026-09-10 16:28:39,150 - preprocessing - INFO - pymovements version: 0.28.0
2026-09-10 16:28:39,157 - preprocessing - INFO - edf2asc version: EDF2ASC version 4.2.1197.0 Linux   standalone Sep 27 2024


In [ ]:
# If you have a specific config file, load it here:
# settings.load_from_yaml("data/MultiplEYE_<...>/multipleye_settings_preprocessing.yaml")

In [2]:
# get the data collection name from the settings and create the path to the data folder
print(f"Active Data Collection: {settings.DATA_COLLECTION_NAME}")
print(f"Dataset Directory: {settings.DATASET_DIR}")

Active Data Collection: MultiplEYE_SV_CH_Zurich_1_2026
Dataset Directory: /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/data/MultiplEYE_SV_CH_Zurich_1_2026


### Inspecting and overriding configuration

After loading the config, you can inspect which sessions are included or excluded, and override these values for the current session without modifying the YAML file.

In [3]:
print(f"Include pilots:  {settings.INCLUDE_PILOTS}")
print(f"Included:        {settings.INCLUDE_SESSIONS}")
print(f"Excluded:        {settings.EXCLUDE_SESSIONS}")
print(f"Output dir:      {settings.OUTPUT_DIR}")
print(f"Run preflight:   {settings.RUN_PREFLIGHT_CHECK}")
print(f"Recalculate:     {settings.RECALCULATE}")

# Override example (uncomment to limit processing to specific sessions):
# settings.INCLUDE_SESSIONS = ["014_DE_DE_1_ET1", "023_DE_DE_1_ET1"]
# settings.EXCLUDE_SESSIONS = []

Include pilots:  True
Included:        ['007_SV_CH_1_ET1', '009_SV_CH_1_ET1']
Excluded:        []
Output dir:      /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026
Run preflight:   False
Recalculate:     False


## MultiplEYE-specific preprocessing & cleaning

In order to be able to run a more generic preprocessing, the MultiplEYE data folder for one language needs to be cleaned and organized in a specific way. Running the script below will:
- unzip session folders if needed
- move session folders from core_sessions folder to the top folder
- check if there is a config file in the stimuli folder (if not, the stimulus folder was probably not uploaded correctly)
- check if there are psychometric tests (if applicable)
	- if necessary, restructure the psychometric test folder.

These steps are very individual for this data collection and results from bugs or changes across the years of collecting data.

Note that executing the cell below for the first time can take very long. However, it will run through quickly after this initial run.

In [4]:
# run the preparation function to prepare the language folder structure
prepare_language_folder()

2026-09-10 16:28:53,186 - preprocessing.scripts.prepare_language_folder - INFO - Stimulus assets unchanged. Skipping copy.


Next, we create a `MultipleyeDataCollection` object from the data folder. This will allow us to easily access the sessions and their information in the next steps.

In [5]:
multipleye = MultipleyeDataCollection.create_from_data_folder(
    settings.DATASET_DIR,
    include_pilots=settings.INCLUDE_PILOTS,
    excluded_sessions=settings.EXCLUDE_SESSIONS,
    included_sessions=settings.INCLUDE_SESSIONS,
)

2026-09-10 16:28:59,838 - preprocessing - INFO - Lab config loaded from /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026/config/config_sv_ch_Zurich_1_2026.py
2026-09-10 16:28:59,839 - preprocessing - INFO - JSON lab config loaded from /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026/config/MultiplEYE_SV_CH_Zurich_1_2026_lab_configuration.json
2026-09-10 16:28:59,841 - preprocessing - INFO - MultipleyeDataCollection initialized. data_root: /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/data/MultiplEYE_SV_CH_Zurich_1_2026/eye-tracking-sessions
2026-09-10 16:28:59,843 - preprocessing - INFO - Main config loaded from /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026/config/config_sv_ch_Zurich_1.py
2026

In [6]:
multipleye.prepare_session_level_information()

Preparing session 007_SV_CH_1_ET1:   0%|          | 0/1 [00:00<?, ?it/s]


KeyError: '007_SV_CH_1_ET1'

In [9]:
multipleye.skipped_sessions.keys()

dict_keys(['007_SV_CH_1_ET1'])

In [6]:
sid = '007_SV_CH_1_ET1'

In [15]:
sess = multipleye.skipped_sessions[sid]

In [7]:
multipleye.skipped_sessions[sid].logfile = multipleye._load_session_logfile(sid)

In [8]:
(multipleye.skipped_sessions[sid].completed_stimuli_ids,
multipleye.skipped_sessions[sid].completed_stimuli_names,
multipleye.skipped_sessions[sid].stimuli_trial_mapping,
) = multipleye._load_session_completed_stimuli(sid)

In [19]:
parsed_answers = preprocessing.parse_answers_from_logfile(sess.logfile, sess.stimuli_trial_mapping)

2026-09-10 16:16:13,057 - py.warnings - PYWARN:WARNING - preprocessing/answers/experiment_log_parser.py:23: UserWarning: ASC messages are missing or empty. Falling back to parsing answers from the experiment logfile. Response time (onset-based) and image offset information will not be available.


In [17]:
question_order_csv = (
                    sess.session_folder_path
                    / "logfiles"
                    / "question_order_versions.csv"
                )

In [21]:
answers_csv = sess.sid.answers_dir / f"{sess.sid}_answers.csv"

In [23]:
source = "logfile"

In [26]:
sess.stimuli_trial_mapping

{'PRACTICE_trial_1': 'Enc_WikiMoon',
 'PRACTICE_trial_2': 'Lit_NorthWind',
 'trial_1': 'Lit_Alchemist',
 'trial_2': 'Lit_Solaris',
 'trial_3': 'Arg_PISACowsMilk',
 'trial_4': 'Lit_BrokenApril',
 'trial_5': 'Lit_MagicMountain',
 'trial_6': 'Ins_HumanRights',
 'trial_7': 'Ins_LearningMobility',
 'trial_8': 'PopSci_MultiplEYE',
 'trial_9': 'PopSci_Caveman',
 'trial_10': 'Arg_PISARapaNui'}

In [24]:
preprocessing.collect_session_answers(
    question_order_csv=question_order_csv,
    stimuli_trial_map=sess.stimuli_trial_mapping,
    stimuli=sess.stimuli,
    parsed_answers=parsed_answers,
    out_path=answers_csv,
    source=source,
    completed_stimuli_ids=sess.completed_stimuli_ids,
)

AttributeError: 'str' object has no attribute 'questions'

In [13]:
from pathlib import Path

### Preflight check

Before processing, run a preflight check to validate the dataset structure and catch common issues (missing files, incorrect folder layout, etc.). In case EDF files are missing, you can use `settings.EXCLUDE_SESSIONS = []` to exclude specific sessions, as shown a few cells above.

In [8]:
preprocessing.run_preflight_check(multipleye)

2026-09-08 16:14:15,788 - preprocessing - INFO - 
  Preflight check — all input files found


## Stage 0: Converting EDF to ASC and Preparing Session-Level Information

Stage 0 refers to the initial steps of preprocessing, which involve converting raw eye-tracking data from its original format (e.g., EDF) into a more accessible format (e.g., ASC), and preparing session-level information. This stage is specific to EyeLink eye-trackers and can be omitted for other eye-trackers.

In [9]:
multipleye.convert_edf_to_asc()

2026-09-08 16:14:19,378 - preprocessing - INFO - Starting EDF to ASC conversion for 1 sessions.
Converting EDF to ASC: 100%|██████████| 1/1 [00:00<00:00, 979.29it/s]
2026-09-08 16:14:19,388 - preprocessing - INFO - EDF to ASC conversion completed.


Once this conversion has been completed, we can load all sessions and parse the .asc files.

In [10]:
multipleye.prepare_session_level_information()

Preparing session 007_SV_CH_1_ET1: 100%|██████████| 1/1 [00:00<00:00,  2.31it/s]


In [11]:
# print an overview on the data collection and the sessions
multipleye

[administrative]
  title: MultiplEYE_SV_CH_Zurich_1_2026
  dataset_type: MultiplEYE
  dataset_description: CH, Zurich (lab 1). SV corpus. Data collected from unknown to unknown.
  number_of_sessions: 0
  number_of_pilots: 1
  number_of_et_sessions_per_participant: 1
  city: Zurich
  lab_number: 1
  country: CH
  tested_language: SV
[language_details]
  metadata_form_exists: False
  language_script: None
  language_family: None
  start_date_of_data_collection: None
  end_date_of_data_collection: None
[data_availability]
  raw_data_available: True
  fixations_available: True
  saccades_available: True
  reading_measures_available: True
[psychometric_tests]
  tests_available: ['LWMC', 'PLAB', 'Flanker', 'Stroop', 'RAN', 'WikiVocab']
[technical_setup]
  eye_tracker_name: EyeLink Portable Duo
  sampling_frequency_hz: 1000
  monitor_name: None
  screen_resolution_width_px: 1920
  screen_resolution_height_px: 1080
[processing]
  preprocessing_date: 2026-09-08
  pipeline_version: 2026.8.28
  n

## Stage 1: Extracting Gaze Samples

In the first preprocessing stage, we extract gaze samples from the .asc files and create a gaze dataframe for each session. This dataframe contains the raw gaze data, including the x and y coordinates of the gaze, the timestamp. We also save the raw gaze data in a separate file for each session.

The next steps are performed for one session only. It is always possible to loop over all sessions and apply the same preprocessing steps to each of them, but for the sake of clarity and simplicity, we will work with one session as an example.



In [7]:
for sess in multipleye.skipped_session_ids:
    print(sess)

007_SV_CH_1_ET1


## Stage 4: Comprehension Question Answers
In addition to gaze data, each session contains answers to comprehension questions.
These are extracted from the ASC messages. The answers are matched to the stimulus order using the `question_order_versions.csv` file in the session's logfiles folder.

In [15]:
answers_csv = sid.answers_dir / f"{sid}_answers.csv"
question_order_csv = (
    sess.session_folder_path / "logfiles" / "question_order_versions.csv"
)


In [16]:
answers_csv

PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/comp_answers/007_SV_CH_1_ET1/007_SV_CH_1_ET1_answers.csv')

In [17]:
question_order_csv

PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/data/MultiplEYE_SV_CH_Zurich_1_2026/eye-tracking-sessions/pilot_sessions/007_SV_CH_1_ET1/logfiles/question_order_versions.csv')

In [16]:
sess.stimuli_trial_mapping

'unknown'

In [10]:
sess = multipleye.skipped_sessions[sid]

In [11]:
parsed_answers = preprocessing.parse_answers_from_logfile(
                            sess.logfile, sess.stimuli_trial_mapping
                        )
source = "logfile"

2026-09-10 15:43:55,093 - py.warnings - PYWARN:WARNING - preprocessing/answers/experiment_log_parser.py:23: UserWarning: ASC messages are missing or empty. Falling back to parsing answers from the experiment logfile. Response time (onset-based) and image offset information will not be available.


In [13]:
sess.sid

Sid(pid='007', lang='SV', country='CH', lab='1', session='ET1', session_id=1, postfix='')

In [12]:
parsed_answers

trial_id,stimulus_name,stimulus_id,question_id,question_onset_ts,preliminary_keys,preliminary_tss,final_confirmation_ts,image_offset_ts,final_answer_key,is_correct,question_stop_ts
str,str,str,str,f64,list[str],list[f64],f64,f64,str,bool,f64
"""trial_1""","""Lit_Alchemist""","""13""","""13121""",null,"[""distractor_a_key"", ""target_key"", ""target_key""]","[277447.4163, 277919.9721, 278109.7405]",280601.2074,null,"""target_key""",true,280604.0878
"""trial_1""","""Lit_Alchemist""","""13""","""13131""",null,"[""distractor_a_key"", ""distractor_c_key"", ""target_key""]","[286030.7483, 286519.0081, 286846.5893]",289352.8486,null,"""target_key""",true,289355.2367
"""trial_2""","""Lit_Solaris""","""7""","""7111""",null,"[""target_key"", ""distractor_b_key"", ""target_key""]","[485182.8381, 486030.3917, 486239.8409]",488120.4612,null,"""target_key""",true,488123.0596
"""trial_2""","""Lit_Solaris""","""7""","""7131""",null,"[""distractor_c_key"", ""target_key"", ""target_key""]","[492894.7184, 493262.0859, 493445.8582]",495008.5204,null,"""target_key""",true,495009.625
"""trial_1""","""Lit_Alchemist""","""4""","""4112""",null,"[""distractor_a_key"", ""distractor_b_key"", ""distractor_a_key""]","[959605.4511, 960991.5427, 961309.8407]",962663.7794,null,"""distractor_a_key""",false,962666.5168
…,…,…,…,…,…,…,…,…,…,…,…
"""trial_10""","""Arg_PISARapaNui""","""11""","""11111""",null,"[""target_key"", ""distractor_c_key""]","[5.5598e6, 5.5606e6]",5.5624e6,null,"""distractor_c_key""",false,5.5624e6
"""trial_10""","""Arg_PISARapaNui""","""11""","""11222""",null,"[""distractor_c_key"", ""target_key"", … ""target_key""]","[5.5656e6, 5.5661e6, … 5.5676e6]",5.5688e6,null,"""target_key""",true,5.5688e6
"""trial_10""","""Arg_PISARapaNui""","""11""","""11121""",null,"[""distractor_a_key"", ""distractor_c_key"", … ""target_key""]","[5.5717e6, 5.5723e6, … 5.5746e6]",5.5759e6,null,"""target_key""",true,5.5759e6


In [14]:
for session_identifier in multipleye.skipped_sessions.keys():
    print(session_identifier)

007_SV_CH_1_ET1


In [20]:
preprocessing.collect_session_answers(
    question_order_csv=question_order_csv,
    stimuli_trial_map=sess.stimuli_trial_mapping,
    stimuli=sess.stimuli,
    parsed_answers=parsed_answers,
    out_path=answers_csv,
    source=source,
    completed_stimuli_ids=sess.completed_stimuli_ids,
)

trial,stimulus,question_id,question_order_version,stimulus_id,snippet_number,condition_number,slot,final_answer_key,answer_text,is_correct,correct_answer_key,correct_answer_text,preliminary_rt_ms,confirmation_rt_ms,preliminary_answer_keys,preliminary_answer_onsets_ms,answer_source
str,str,str,i64,i32,i32,i64,str,str,str,bool,str,str,f64,f64,list[str],list[f64],str
"""trial_1""","""Lit_Alchemist""","""4112""",2,4,1,1,"""local_question_1""","""distractor_a_key""","""Taket.""",false,"""target_key""","""Stjärnorna.""",null,null,"[""distractor_a_key"", ""distractor_b_key"", ""distractor_a_key""]",null,"""logfile"""
"""trial_1""","""Lit_Alchemist""","""4111""",2,4,1,1,"""local_question_2""","""target_key""","""För att de fungerade bättre so…",true,"""target_key""","""För att de fungerade bättre so…",null,null,"[""distractor_a_key"", ""distractor_c_key"", … ""target_key""]",null,"""logfile"""
"""trial_1""","""Lit_Alchemist""","""4121""",2,4,1,2,"""bridging_question_1""","""target_key""","""Han ville hålla fåren inomhus.…",true,"""target_key""","""Han ville hålla fåren inomhus.…",null,null,"[""distractor_c_key"", ""distractor_a_key"", … ""target_key""]",null,"""logfile"""
"""trial_1""","""Lit_Alchemist""","""4122""",2,4,1,2,"""bridging_question_2""","""target_key""","""I en kyrka.""",true,"""target_key""","""I en kyrka.""",null,null,"[""target_key""]",null,"""logfile"""
"""trial_1""","""Lit_Alchemist""","""4131""",2,4,1,3,"""global_question_1""","""target_key""","""Fåren har en ömsesidig vänskap…",true,"""target_key""","""Fåren har en ömsesidig vänskap…",null,null,"[""distractor_c_key"", ""target_key""]",null,"""logfile"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""trial_10""","""Arg_PISARapaNui""","""11111""",4,11,1,1,"""local_question_2""","""distractor_c_key""","""I början av maj. """,false,"""target_key""","""För nio månader sedan.""",null,null,"[""target_key"", ""distractor_c_key""]",null,"""logfile"""
"""trial_10""","""Arg_PISARapaNui""","""11222""",4,11,2,2,"""bridging_question_1""","""target_key""","""Utarmningen av naturresurser.""",true,"""target_key""","""Utarmningen av naturresurser.""",null,null,"[""distractor_c_key"", ""target_key"", … ""target_key""]",null,"""logfile"""
"""trial_10""","""Arg_PISARapaNui""","""11121""",4,11,1,2,"""bridging_question_2""","""target_key""","""Varför de stora träden hade fö…",true,"""target_key""","""Varför de stora träden hade fö…",null,null,"[""distractor_a_key"", ""distractor_c_key"", … ""target_key""]",null,"""logfile"""


In [21]:
gaze = preprocessing.load_trial_level_raw_data(
                sess.sid,
                trial_columns=settings.TRIAL_COLS,
                load_metadata=True,
            )

In [22]:
parsed_answers = preprocessing.parse_answers_from_messages(
                            gaze.messages
                        )
source = "asc"

In [23]:
parsed_answers

trial_id,stimulus_name,stimulus_id,question_id,question_onset_ts,preliminary_keys,preliminary_tss,final_confirmation_ts,image_offset_ts,final_answer_key,is_correct,question_stop_ts
str,str,str,str,f64,list[str],list[f64],f64,f64,str,bool,f64
"""PRACTICE_trial_1""","""Enc_WikiMoon""","""13""","""13121""",1.960056e6,"[""distractor_a_key"", ""target_key"", ""target_key""]","[1.969264e6, 1.969736e6, 1.969926e6]",1.972418e6,1.972418e6,"""target_key""",true,1.972419e6
"""PRACTICE_trial_1""","""Enc_WikiMoon""","""13""","""13131""",1.972419e6,"[""distractor_a_key"", ""distractor_c_key"", ""target_key""]","[1.977847e6, 1.978336e6, 1.978663e6]",1.981169e6,1.981169e6,"""target_key""",true,1.98117e6
"""PRACTICE_trial_2""","""Lit_NorthWind""","""7""","""7111""",2.172482e6,"[""target_key"", ""distractor_b_key"", ""target_key""]","[2.176997e6, 2.177845e6, 2.178054e6]",2.179935e6,2.179935e6,"""target_key""",true,2.179935e6
"""PRACTICE_trial_2""","""Lit_NorthWind""","""7""","""7131""",2.179936e6,"[""distractor_c_key"", ""target_key"", ""target_key""]","[2.184709e6, 2.185076e6, 2.18526e6]",2.186821e6,2.186821e6,"""target_key""",true,2.186822e6
"""trial_1""","""Lit_Alchemist""","""4""","""4112""",2.648178e6,"[""distractor_a_key"", ""distractor_b_key"", ""distractor_a_key""]","[2.651415e6, 2.652801e6, 2.653119e6]",2.654473e6,2.654473e6,"""distractor_a_key""",false,2.654475e6
…,…,…,…,…,…,…,…,…,…,…,…
"""trial_10""","""Arg_PISARapaNui""","""11""","""11111""",7.249029e6,"[""target_key"", ""distractor_c_key""]","[7.251601e6, 7.25233e6]",7.254172e6,7.254172e6,"""distractor_c_key""",false,7.254173e6
"""trial_10""","""Arg_PISARapaNui""","""11""","""11222""",7.254173e6,"[""distractor_c_key"", ""target_key"", … ""target_key""]","[7.257411e6, 7.257834e6, … 7.25937e6]",7.260523e6,7.260523e6,"""target_key""",true,7.260524e6
"""trial_10""","""Arg_PISARapaNui""","""11""","""11121""",7.260525e6,"[""distractor_a_key"", ""distractor_c_key"", … ""target_key""]","[7.263465e6, 7.264081e6, … 7.266363e6]",7.267707e6,7.267707e6,"""target_key""",true,7.267708e6
